<a href="https://colab.research.google.com/github/NewTopia1220/Java-Spring/blob/main/%EB%8D%B0%EC%9D%B4%ED%84%B0_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

os -->파이썬에 기본적으로 내장된 운영체제(OS) 제어용 모듈을 불러온다

asyncio --> 파이썬에 기본 내장된 비동기 프로그래밍 모듈 불러옴

load_dotenv --> api키 노출이 위험한 저오들을 .env라는 파일에 저장해서 load_dptenv는 파일에 적힌 설정값들을 파이썬프로그램 안으로 안전하게 불러옴

ArticleProcessor -->뉴스 기사의 텍스트를 다듬거나 분석하는 도구

run_pipeline_async -->기사를 수집하고 처리하는 일련의 과정을 비동기 방식으로 실행할수있는 함수

위에 2개 함수는 직접 만든 함수입니다




In [ ]:
import os
import asyncio
from dotenv import load_dotenv
from news_common import ArticleProcessor, run_pipeline_async

In [ ]:
카테고리 별로 나눈것 <하드코딩?>

In [ ]:
SECTOR_CONFIG = {
    "반도체/AI": {
        "search_query": "반도체 AI",
        "keywords": ["삼성전자", "SK하이닉스", "하이닉스", "HBM", "반도체", "AI", "엔비디아",
                     "D램", "낸드", "파운드리", "TSMC", "NPU", "GPU", "EUV", "데이터센터"]
    },
    "2차전지": {
        "search_query": "2차전지 배터리",
        "keywords": ["에코프로", "에너지솔루션", "양극재", "배터리", "리튬", "전기차", "테슬라",
                     "음극재", "전고체", "LFP", "NCM", "IRA", "캐즘", "CATL", "BYD"]
    },
    "바이오": {
        "search_query": "제약 바이오",
        "keywords": ["셀트리온", "바이오로직스", "신약", "임상", "제약", "바이오", "FDA",
                     "유한양행", "알테오젠", "HLB", "바이오시밀러", "항암제", "기술수출", "CDMO"]
    },
    "금융/밸류업": {
        "search_query": "금융주 밸류업",
        "keywords": ["KB금융", "신한지주", "하나금융지주", "우리금융", "메리츠", "은행",
                     "주주환원", "배당", "자사주", "소각", "PBR", "ROE", "밸류업"]
    },
    "방산": {
        "search_query": "K방산 무기 수출",
        "keywords": ["한화에어로", "현대로템", "방산", "수출", "미사일", "무기", "폴란드",
                     "KAI", "LIG넥스원", "한화시스템", "K9", "자주포", "전투기", "수주잔고"]
    },
    "IT/플랫폼": {
        "search_query": "네이버 카카오 플랫폼",
        "keywords": ["네이버", "카카오", "크래프톤", "엔씨소프트", "플랫폼", "클라우드",
                     "게임", "웹툰", "라인", "카카오톡", "카카오페이", "넥슨"]
    },
    "엔터/미디어": {
        "search_query": "엔터 아이돌 K팝",
        "keywords": ["하이브", "JYP", "에스엠", "와이지", "아이돌", "팬덤", "콘서트",
                     "K팝", "빌보드", "CJ ENM", "방탄소년단", "BTS", "뉴진스"]
    },
    "자동차/모빌리티": {
        "search_query": "현대차 기아 전기차",
        "keywords": ["현대차", "현대자동차", "기아", "모비스", "현대모비스",
                     "전기차", "하이브리드", "자율주행", "제네시스", "아이오닉"]
    }
}



<기업별로 필터링 한것들>

In [ ]:
COMPANY_CONFIG = {
    "삼성전자 (IT/반도체)": {
        "search_query": "삼성전자",
        "core_keywords": ["삼성전자", "삼전"],
        "sub_keywords": ["이재용", "파운드리", "갤럭시", "메모리", "HBM", "반도체",
                         "엔비디아", "TSMC", "D램", "낸드", "실적", "매출", "영업이익",
                         "주가", "목표가", "배당", "외국인", "투자"]
    },
    "LG에너지솔루션 (2차전지)": {
        "search_query": "LG에너지솔루션",
        "core_keywords": ["LG에너지솔루션", "엔솔"],
        "sub_keywords": ["LG화학", "권영수", "김동명", "배터리", "원통형", "테슬라",
                         "전기차", "캐즘", "수주", "보조금", "실적", "매출", "영업이익",
                         "주가", "목표가", "배당", "외국인", "투자"]
    },
    "삼성바이오로직스 (제약/바이오)": {
        "search_query": "삼성바이오로직스",
        "core_keywords": ["삼성바이오로직스", "삼바"],
        "sub_keywords": ["존림", "CDMO", "위탁생산", "바이오시밀러", "에피스", "FDA",
                         "임상", "신약", "승인", "공장", "실적", "매출", "영업이익",
                         "주가", "목표가", "배당", "외국인", "투자"]
    },
    "현대차 (자동차/모빌리티)": {
        "search_query": "현대차",
        "core_keywords": ["현대차", "현대자동차"],
        "sub_keywords": ["정의선", "제네시스", "아이오닉", "전기차", "하이브리드",
                         "HEV", "수출", "노조", "점유율", "판매량", "실적", "매출",
                         "영업이익", "주가", "목표가", "배당", "외국인", "투자"]
    },
    "네이버 (IT/플랫폼)": {
        "search_query": "네이버",
        "core_keywords": ["네이버", "NAVER"],
        "sub_keywords": ["최수연", "이해진", "라인", "하이퍼클로바", "웹툰", "플랫폼",
                         "AI", "클라우드", "광고", "커머스", "실적", "매출", "영업이익",
                         "주가", "목표가", "배당", "외국인", "투자"]
    },
    "KB금융 (금융/밸류업)": {
        "search_query": "KB금융",
        "core_keywords": ["KB금융", "국민은행"],
        "sub_keywords": ["양종희", "리딩뱅크", "주주환원", "배당", "저PBR", "밸류업",
                         "자사주", "소각", "금리", "홍콩", "실적", "매출", "영업이익",
                         "주가", "목표가", "외국인", "투자"]
    },
    "한화에어로스페이스 (방산/우주항공)": {
        "search_query": "한화에어로스페이스",
        "core_keywords": ["한화에어로스페이스", "한화에어로"],
        "sub_keywords": ["김동관", "방산", "수출", "폴란드", "자주포", "전투기", "K9",
                         "무기", "수주", "누리호", "실적", "매출", "영업이익",
                         "주가", "목표가", "배당", "외국인", "투자"]
    },
    "하이브 (엔터/미디어)": {
        "search_query": "하이브",
        "core_keywords": ["하이브", "HYBE"],
        "sub_keywords": ["방시혁", "민희진", "BTS", "방탄소년단", "뉴진스", "세븐틴",
                         "아이돌", "빌보드", "팬덤", "콘서트", "실적", "매출", "영업이익",
                         "주가", "목표가", "배당", "외국인", "투자"]
    }
}


위에서 import한 ArticleProcessor함수 코드르 를 분석해보면 proc의 타입은 ArticleProcessor이어야하고


title 은 str 타입이고 text는  str타입 그리고 info는 dict 이어야함 그리고 --> bool은 결과값은 true나 false가 나와야함


인자로 클래스 타입으로 받아서 return할때 클래스 안에있는 함수들을 쓸수있는데 아까 위에서import한 ArticleProcessor을 통해 접근 가능하다 추후 나오는 함수에 대해선 설명하겠음

title과 text info 를 넘겨서 필터링 한다고 생각하면 쉽다

In [ ]:
def verify_sector_fn(proc: ArticleProcessor, title: str, text: str, info: dict) -> bool:
    return proc.verify_sector(title, text, info["keywords"])

def verify_company_fn(proc: ArticleProcessor, title: str, text: str, info: dict) -> bool:
    return proc.verify_company(title, text, info["core_keywords"], info["sub_keywords"])

이 함수를 알기전에 비동기와 동기에대해서 간단하게 설명하겠음 -->동기는 예를 들어서 커피숍에서 커피를 주문하면 나올때까지 카운터 앞에서 기다리고
비동기는 커피숍에서 커피를 주문하고 자리로 앉아서 다른 할일 가능 이라고 생각하면 된다

async--> 비동기 함수 선언

os가 .getenv를 하게되면 네이버 키와 그록 키를 읽어오고 하나라도 없으면 함수를 종료함

await asyncio.gather --> 함수 문법인데 비동기 라이브러리 안의 gather함수를 호출임 그리고 비동기 함수니까 await(여기서 기다리는동안 다른 작업해도되는 키워드)

이제 함수를 이용해서 비동기 파이프라인을 가동시킨다

함수안에 위에서 정의한 SECTOR_CONFIG를 써주고 verify_sector_fn 위에서 정의한 함수도 넘겨주고
DB table도 넘겨주고  Max_per_Setor=100은 네이버 api가 100개 목록을 반환하면 필터링을 적용해서 저장을 하는데 만약 100개가 넘는다면(그럴일은 없겠지만) 딱 100개만 저장함

그리고 API키를 넘겨줌

In [ ]:
async def main():
    if not os.getenv("NAVER_CLIENT_ID") or not os.getenv("NAVER_CLIENT_SECRET"):
        print("네이버 API 키 없음.")
        return
    if not os.getenv("GROQ_API_KEY_1"):
        print("Groq API 키 없음.")
        return

    print("=" * 60)
    print("  섹터별 + 기업별 뉴스 수집 동시 시작")
    print("=" * 60)

    sec_count, comp_count = await asyncio.gather(
        run_pipeline_async(
            sector_config=SECTOR_CONFIG,
            verify_fn=verify_sector_fn,
            db_table="news_data_sec",
            max_per_sector=100,
            groq_api_keys=[
                os.getenv("GROQ_API_KEY_1")
            ],
        ),
        run_pipeline_async(
            sector_config=COMPANY_CONFIG,
            verify_fn=verify_company_fn,
            db_table="news_data",
            max_per_sector=100,
            groq_api_keys=[
                os.getenv("GROQ_API_KEY_1")

            ]
        ),
    )

    print("=" * 60)
    print(f"  최종 완료 — news_data_sec: {sec_count}건 | news_data: {comp_count}건")
    print("=" * 60)


import os: 파일 경로, 폴더 관리, 환경 변수 등 운영체제 기능을 제어할 때 씁니다.
----------------------------------------------------
import re: 정규표현식을 이용해 텍스트에서 이메일, 전화번호 등 원하는 패턴만 찾거나 바꿀 때 씁니다.
----------------------------------------------------
import json: JSON 형식의 데이터를 파이썬에서 읽고 쓰거나 다룰 때 사용합니다.
-----------------------------------------------------
import asyncio: 네트워크 통신같이 대기 시간이 긴 작업들을 멈춤 없이 동시에(비동기) 처리할 때 씁니다.
-----------------------------------------------------
import joblib: 학습이 끝난 머신러닝 모델이나 파이썬 데이터를 파일로 저장하고 빠르게 불러올 때 씁니다.
-----------------------------------------------------
import threading: 프로그램 내에서 여러 작업을 동시에 실행(멀티스레딩)하여 작업 속도를 높일 때 사용합니다.
-----------------------------------------------------
import aiohttp: 비동기 방식으로 엄청 빠르게 여러 웹페이지를 가져오거나 API를 호출할 때 씁니다
-----------------------------------------------------
import oracledb: 파이썬 코드를 오라클(Oracle) 데이터베이스와 연결해 데이터를 넣고 뺄 때 사용합니다.
-----------------------------------------------------
import torch: 파이토치(PyTorch) 기본 라이브러리로, 딥러닝을 위한 수학/텐서 연산을 할 때 씁니다.
-----------------------------------------------------
import torch.nn as nn: 딥러닝 신경망(Neural Network)의 뼈대(레이어)를 조립하고 모델을 만들 때 사용합니다
-----------------------------------------------------
import torch.nn.functional as F: 딥러닝 계산에 필요한 수학 함수(활성화 함수, 손실 함수 등)를 사용할 때 씁니다.
-----------------------------------------------------
from concurrent.futures import ThreadPoolExecutor: 여러 개의 스레드를 풀(Pool)로 묶어서 병렬 작업을 훨씬 쉽고 안전하게 관리할 때 씁니다.
-----------------------------------------------------
from collections import Counter: 데이터 목록에서 특정 단어나 항목이 몇 번 등장했는지 개수를 척척 셀 때 씁니다.
-----------------------------------------------------
from bs4 import BeautifulSoup: 웹 크롤링을 할 때 복잡한 HTML 문서에서 원하는 텍스트만 예쁘게 뽑아낼 때 사용합니다.
-----------------------------------------------------
from konlpy.tag import Okt: 한국어 문장을 분석해서 명사, 동사 같은 형태소 단위로 쪼갤 때(자연어 처리) 씁니다.
-----------------------------------------------------
from datetime import datetime: 현재 시간을 구하거나 날짜 계산을 하는 등 시간 데이터를 다룰 때 씁니다.
-----------------------------------------------------
from openai import AsyncOpenAI: 챗GPT 같은 OpenAI의 AI 모델을 멈춤 없이 비동기 방식으로 호출할 때 사용합니다.
-----------------------------------------------------
from transformers import BertModel, BertTokenizerFast: 허깅페이스의 강력한 자연어 딥러닝 모델(BERT)과 텍스트 변환기(토크나이저)를 가져올 때 씁니다.
-----------------------------------------------------

In [ ]:
import os
import re
import json
import asyncio
import joblib
import threading
import aiohttp
import oracledb
import torch
import torch.nn as nn
import torch.nn.functional as F
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
from bs4 import BeautifulSoup
from konlpy.tag import Okt
from datetime import datetime
from openai import AsyncOpenAI
from transformers import BertModel, BertTokenizerFast

공통 필터 단어

In [ ]:
EXCLUDE_WORDS = [
    "사형", "징역", "검찰", "구속", "당대표", "정당", "비서관", "정치", "의원",
    "대통령", "국회", "여당", "야당", "선거", "총선", "대선", "경찰", "수사",
    "장관", "공직자", "고위직", "재산", "윤리위원회",
    "다주택", "청약", "아파트", "전세", "부동산", "분양"
]

TITLE_EXCLUDE_WORDS = [
    "시황", "마감", "특징주", "인사이트", "증시", "코스피", "코스닥", "종합",
    "순매수", "순매도", "동반 상승", "동반 하락", "외인", "기관", "뉴욕증시",
    "관련주", "테마주", "수혜주", "급등주", "종목추천"
]


Bert 모델 구조 -->BertBaseModel클래스 정의를한다 ( 이런 구조의 모델이야 kykim/bert-kor-base 설계도를 가져와서 뼈대를 만들고 cls 분류층을 정의

Pytorch의 정의인데 model을 호출하면 자동으로 forward가 실행된다 forward로 데이터 들이 들어가서 학습이됨

In [ ]:
class BertBaseModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained("kykim/bert-kor-base")
        self.cls  = nn.Linear(768, 4)

    def forward(self, input_ids, attention_mask):
        return self.cls(
            self.bert(input_ids=input_ids, attention_mask=attention_mask)[1]
        )

싱글톤 패턴

cls<-->self --> self는 만들어진 객채 자신 / cls --> 클래스 그 자체

가장 조상인 object조상의 객채 생성 능력을 빌려와서 ModelManger클래스 모양대로 메모리에 공간을 하나 만듬
그리고 instance에 객채를 만듬

지금 만든 객채에 initialized 이름표 강제로 만들고 false라는 값을 넣어줌 초기화가 안됨 그리고 instance넘겨줌

그리고 init으로 넘어가서 접근이 가능함 self를쓴다
이제 로지스틱모델 버트 모델 불러온다


In [ ]:
class ModelManager:

    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._initialized = False
        return cls._instance

    def __init__(self):
        if self._initialized:
            return
        self._initialized = True
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"장치: {self.device}")
        self._load_logistic()
        self._load_bert()

    def _load_logistic(self):
        print("로지스틱 회귀 모델 로딩")
        try:
            self.lr_model      = joblib.load('clickbait_model.joblib')
            self.lr_vectorizer = joblib.load('tfidf_vectorizer.joblib')
            print("로지스틱 회귀 로딩 완료")
        except Exception as e:
            print(f"로딩 실패: {e}")
            self.lr_model      = None
            self.lr_vectorizer = None

    def _load_bert(self):
        try:
            import __main__
            __main__.BertBaseModel = BertBaseModel
            self.bert_model = torch.load(
                './model.pt', map_location=self.device, weights_only=False
            )
            self.bert_model.to(self.device)  #중복임 근데 관례라고함
            self.bert_model.eval()
            self.tokenizer = BertTokenizerFast.from_pretrained("kykim/bert-kor-base")
            print("BERT 로딩 완료")
        except Exception as e:
            print(f" BERT 로딩 실패: {e}")
            self.bert_model = None
            self.tokenizer  = None


본문 비동기 수집 / 텍스트 정제 / BERT 추론 / 필터링

threading.Lock() 프로그램의 속도를 높이려고 여러 작업이 동시에 뉴스 기사들을 분석하려고하는데 지금 우리가쓰는 okt는 내부적으로 자바를땡겨와서 사용하는데 동시에 여러 명이 말을 걸면 에러를 뿜으면서 애러를낸다 그리서 해결책으로 threading.Lock() 으로 열쇠를 만든다 이 규칙을 정해주면 스레드 들이 엉키지 않고 한번에 하나씩 안전하게 한국어 분석을 마칠수 있게 해줌


비동기 본문 수집에서 인자들을 보면 함수앞에 async가 붙어있다 비동기 함수로 비동기를 사용해서 본문 수집을 한다는거다
aiohttp.ClientSession방식은 기존에는 request.get을 사용했다면 지금은 이방식을 상요한다 커넥션 풀링(길뚫어둔거 재사용하기)/쿠키와 설정공유(내 신분증 유지하기) /비동기 처리 ( aiohttp가 비동기 통신을 위해 만들어졌기 때문에 세션 안에서 수백개의 인터넷 주소에 동시에 url요청을 날려놓고 응답이 빠르게 오는것부터 처리가 가능하다

header에 봇이 아니라고 적어주고 가짜 신분증만듬
timeout --> 5초만에 응답안오면 종료
밑에 session.get 아까 만들어온 세션을 통해 url로 위장 신분증을 들고 접속함


html --> 페이지의 전체 소스 코드를 다운로드한다 await을 쓰면 다 다운받을때까지 시간걸리니까 다른 작업하라고 한다
html이 되면 BeautifulSoup하면서 파싱한다
content 에 본문내용 추력

content.text --> 하면 글자만 추출됨 태그들 다 지워지고 " " 태그자리에는 이렇게 넣어준다 strip true는 알맹이만 엔터나 줄바꿈 공백들을 가위로 자르듯 깔끔하게 삭제

In [ ]:
class ArticleProcessor:


    def __init__(self, model_manager: ModelManager):
        self.mm        = model_manager
        self.okt       = Okt()
        self._okt_lock = threading.Lock()  # KoNLPy 스레드 비안전 → 락

    # ------ 비동기 본문 수집 ------
    async def fetch_content(self, session: aiohttp.ClientSession, url: str) -> str | None:
        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            timeout = aiohttp.ClientTimeout(total=5)
            async with session.get(url, headers=headers, timeout=timeout) as resp:
                if resp.status != 200:
                    return None
                html    = await resp.text()
                soup    = BeautifulSoup(html, "html.parser")
                content = soup.select_one("#dic_area, #newsct_article, #articeBody")
                if content:
                    text = content.get_text(" ", strip=True)
                    return text if text else None
                return None
        except Exception:
            return None

# 전처리

re.sub(정규표현식,정규표현식 형식있으면 바꿀꺼,원본)
strip -->맨 앞과 맨 뒤에 남은 쓸데없는 여백을 자름
split --> 문단별로 잘라줌
글자가 20개 이하인건 삭제

if (용량 이내): 가방에 계속 담는

else (용량 초과): 지금까지 담은 가방은 창고에 넣고 새 가방에 지금 물건을 담는다

마지막 if: 모든 일이 끝난 뒤, 손에 들고 있던 마지막 가방까지 창고에 넣고 퇴근한다

In [ ]:
    def clean_and_split(self, full_text: str, max_chars: int = 100) -> list:
        if not full_text:
            return []
        text = re.sub(r'[a-zA-Z0-9+-_.]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', full_text)
        text = re.sub(r'©.*|Copyright.*|무단\s*전재.*|배포\s*금지.*', '', text)
        text = re.sub(r'\S+\s*기자', '', text)
        text = re.sub(r'https?://\S+', '', text)
        text = re.sub(r'\[.*?\]|\(.*?사진.*?\)|\(.*?제공.*?\)', '', text)
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n{2,}', '\n', text)
        text = text.strip()

        sentences_raw = re.split(r'(?<=[다요음임])\.\s+|\n', text)
        sentences     = [s.strip() for s in sentences_raw if len(s.strip()) > 20]
        if not sentences:
            return []

        chunks, current_chunk = [], ""
        for sentence in sentences:
            if len(current_chunk) + len(sentence) <= max_chars:
                current_chunk += (" " + sentence) if current_chunk else sentence
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = sentence
        if current_chunk:
            chunks.append(current_chunk.strip())
        return chunks

mm = self.mm: 앞서 만든 ModelManager를 짧게 mm으로 부릅니다.

if not paragraphs or mm.bert_model is None:: 입력받은 글이 없거나 AI 모델이 로딩 안 됐으면,

return "알 수 없음", 0.0: "알 수 없음"과 확률 0점을 주고 끝냅니다.

class_names = ["사실형", "예측형", "대화형", "추론형"]: AI가 분류할 4가지 카테고리 명단입니다.

predicted_labels = []: 각 덩어리별로 예측한 결과(이름)를 담을 리스트입니다.

chunk_confidences = []: 각 덩어리별로 AI가 얼마나 확신했는지(확률)를 담을 리스트입니다.

all_probs_per_chunk = []: 각 덩어리별 4개 카테고리의 모든 확률 점수를 보관합니다.

mm.bert_model.eval(): 모델을 "평가 모드"로 바꿉니다. (학습 때 쓰는 기능들을 끕니다.)

with torch.no_grad():: "기울기 계산 안 함" 설정입니다. (메모리를 아끼고 속도를 높입니다.)

for p in paragraphs:: 아까 clean_and_split으로 나눈 덩어리들을 하나씩 꺼냅니다.

inputs = mm.tokenizer(...): 텍스트를 AI가 이해할 수 있는 숫자(토큰)로 변환합니다. (길이 128자 제한)

input_ids = ... / attention_mask = ...: 변환된 숫자 데이터를 GPU/CPU 장치로 보냅니다.

outputs = mm.bert_model(...): [핵심] AI 모델에 데이터를 넣어 결과를 뽑습니다.

probs = F.softmax(...).squeeze().tolist(): 결과 숫자를 0~1 사이의 확률(점수)로 변환합니다.

max_idx = probs.index(max(probs)): 4개 점수 중 가장 높은 점수의 번호를 찾습니다.

predicted_labels.append(...): 가장 높은 점수의 카테고리 이름을 저장합니다.

chunk_confidences.append(...): 그 카테고리의 확신 점수를 저장합니다.

all_probs_per_chunk.append(probs): 전체 확률 분포를 저장합니다

label_counts = Counter(predicted_labels): 각 카테고리가 몇 번씩 나왔는지 셉니다. (예: 사실형 3번, 예측형 1번)

top_count = ...: 가장 많이 나온 투표수를 찾습니다. (예: 3회)

top_labels = ...: 가장 많이 투표받은 카테고리 목록을 만듭니다. (동점이 있을 수 있으니까요.)

if len(top_labels) == 1:: 1등이 압도적이면 바로 그놈을 final_label로 정합니다.

else:: 만약 동점(예: 사실형 2회, 예측형 2회)이라면,

avg_probs = ...: 모든 덩어리의 확률 점수를 평균 내서 더 높은 평균값을 가진 타입을 선택합니다.

win_confs = ...: 최종 선택된 타입에 투표한 덩어리들의 확신 점수만 모읍니다.

if not win_confs: return final_label, 0.0: 혹시나 데이터가 없으면 0점 처리합니다.

vote_ratio = len(win_confs) / len(predicted_labels): [중요] 전체 덩어리 중 최종 타입이 뽑힌 비율입니다. (일관성을 체크합니다.)

final_prob = (sum(...) / len(...)) * vote_ratio * 100: (평균 확신도 × 투표 비율)을 계산해 100점 만점으로 환산합니다.

return final_label, round(final_prob, 1): 최종 타입과 소수점 첫째 자리까지의 점수를 돌려줍니다.

승자 명단 추출: 최종 1등으로 뽑힌 유형에 투표한 덩어리들의 '확신 점수'만 따로 모읍니다.

전체 찬성률 계산: 전체 덩어리 중 몇 개가 1등 의견에 동의했는지 비율(찬성수 ÷ 전체수)을 구합니다.

신뢰도 점수 합산: '찬성파들의 평균 점수'에 '찬성률'을 곱해, 의견이 일치할수록 높은 점수가 나오게 만듭니다.

최종 결과 보고: 확정된 뉴스 유형과 100점 만점으로 환산한 최종 신뢰도를 소수점 첫째 자리까지 반환합니다.

In [ ]:
    def predict_type(self, paragraphs: list) -> tuple[str, float]:
        mm = self.mm
        if not paragraphs or mm.bert_model is None:
            return "알 수 없음", 0.0

        class_names         = ["사실형", "예측형", "대화형", "추론형"]
        predicted_labels    = []
        chunk_confidences   = []
        all_probs_per_chunk = []

        mm.bert_model.eval()
        with torch.no_grad():
            for p in paragraphs:
                inputs = mm.tokenizer(
                    p, return_tensors="pt",
                    max_length=128, truncation=True, padding="max_length"
                )
                input_ids      = inputs['input_ids'].to(mm.device)
                attention_mask = inputs['attention_mask'].to(mm.device)
                outputs        = mm.bert_model(input_ids, attention_mask)
                probs          = F.softmax(outputs, dim=1).squeeze().tolist()
                max_idx        = probs.index(max(probs))
                predicted_labels.append(class_names[max_idx])
                chunk_confidences.append(max(probs))
                all_probs_per_chunk.append(probs)

        label_counts = Counter(predicted_labels)
        top_count    = label_counts.most_common(1)[0][1]
        top_labels   = [l for l, c in label_counts.items() if c == top_count]

        if len(top_labels) == 1:
            final_label = top_labels[0]
        else:
            avg_probs   = [sum(col) / len(col) for col in zip(*all_probs_per_chunk)]
            final_label = class_names[avg_probs.index(max(avg_probs))]

        win_confs  = [c for l, c in zip(predicted_labels, chunk_confidences)
                      if l == final_label]

        if not win_confs:  # 만약 결과 리스트가 비어있다면
            return final_label, 0.0

        vote_ratio = len(win_confs) / len(predicted_labels)
        final_prob = (sum(win_confs) / len(win_confs)) * vote_ratio * 100
        return final_label, round(final_prob, 1)


verify_sector (섹터용)
"빈 본문/금지어 걸러내고 → 제목에 키워드 있으면 바로 Pass → 제목에 없으면 본문에 관련 단어 2개 이상 있어야 합격!"

verify_company (기업용)
"빈 본문/금지어 걸러내고 → 제목에 기업명(Core) 없으면 무조건 탈락 → 제목에 기업명 있으면 본문에 관련 단어(Sub) 1개 이상은 있어야 최종 합격!"

In [ ]:
  # ------ 섹터 필터 (동기 → run_in_executor로 호출) ------
    def verify_sector(self, title: str, text: str, keywords: list) -> bool:
        """CrawlingNaverApi용: keywords 단일 리스트"""
        if not text:
            return False
        for bad in TITLE_EXCLUDE_WORDS:
            if bad in title:
                return False
        for bad in EXCLUDE_WORDS:
            if bad in title or bad in text:
                return False
        if any((kw in title) for kw in keywords):
            return True
        with self._okt_lock:
            nouns = self.okt.nouns(text)
        return sum(1 for n in nouns if n in keywords) >= 2

    def verify_company(
        self, title: str, text: str, core_keywords: list, sub_keywords: list
    ) -> bool:
        """CrawlingNaverTOP7용: core + sub keywords"""
        if not text:
            return False
        for bad in TITLE_EXCLUDE_WORDS:
            if bad in title:
                return False
        for bad in EXCLUDE_WORDS:
            if bad in title or bad in text:
                return False
        if not any(core in title for core in core_keywords):
            return False
        with self._okt_lock:
            nouns = self.okt.nouns(text)
        return sum(1 for n in nouns if n in sub_keywords) >= 1


"여러 개의 API 키를 번갈아 쓰면서(비동기), 기사를 800자만 읽고, 호재/악재 판별과 1줄 요약을 JSON 형태로 안전하게 뽑아내는 AI 공장"

async def 와 await

이유: AI 서버의 응답을 기다리는 동안 프로그램이 멈추지 않고, 다른 기사들도 동시에 AI에게 던져서 처리 속도를 극대화합니다.

await client...create(...) -->"이 작업은 오래 걸리니까 일단 던져놓고, 결과가 나올 때까지 이 자리에서 대기해!" (그동안 프로그램의 다른 부분은 계속 돌아감

In [ ]:
class GroqAnalyzer:
    def __init__(self, api_keys: str):

        if api_keys is None:
            api_keys = [os.getenv("GROQ_API_KEY")]
        elif isinstance(api_keys, str):
            api_keys = [api_keys]
        self._clients = [
            AsyncOpenAI(base_url="https://api.groq.com/openai/v1", api_key=k, timeout=20.0)
            for k in api_keys if k
        ]
        self._idx = 0
        self._system_prompt = (
            "너는 주식 시장 뉴스 분석 전문가다.\n"
            "1. 반드시 유효한 JSON 형식으로만 응답한다.\n"
            "2. 'sentiment'는 [호재, 악재, 중립] 중 하나로만 선택한다.\n"
            "   - 호재: 실적 상승, 수주 성공, 신제품 흥행, 목표주가 상향 등\n"
            "   - 악재: 실적 하락, 소송, 리콜, 규제, 목표주가 하향 등\n"
            "   - 중립: 단순 사실 전달, 영향 미미, 호악재 혼재 등\n"
            "3. 'summary'는 1문장으로 핵심만 요약한다.\n"
            "4. 키는 'summary'와 'sentiment'만 사용한다."
        )

    async def analyze(self, title: str, full_text: str) -> dict:
        if not full_text:
            return {"summary": "본문 없음", "sentiment": "중립"}
        # 라운드로빈: 요청마다 다음 키 사용
        client = self._clients[self._idx % len(self._clients)]
        self._idx += 1
        try:
            response = await client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": self._system_prompt},
                    {"role": "user",
                     "content": f"제목: {title}\n본문: {full_text[:800]}"}
                ],
                temperature=0.5,
                response_format={"type": "json_object"},
                max_tokens=2048
            )
            result = json.loads(response.choices[0].message.content.strip())
            if result.get("sentiment") not in ["호재", "악재", "중립"]:
                mapping = {"긍정": "호재", "부정": "악재"}
                result["sentiment"] = mapping.get(result.get("sentiment"), "중립")
            return result
        except Exception as e:
            print(f"Groq 오류 (키 #{self._idx % len(self._clients)}): {e}")
            return {"summary": "요약 오류", "sentiment": "중립"}


이건그냥 db저장

In [ ]:
class DatabaseManager:
    LIB_DIR     = r'C:\Oracle\dbhomeXE\bin'
    WALLET_PATH = r'C:\Users\PC\Desktop\Wallet_Stoxle'

    def connect(self):
        os.environ['TNS_ADMIN'] = self.WALLET_PATH
        oracledb.init_oracle_client(lib_dir=self.LIB_DIR)
        return oracledb.connect(
            user="ADMIN", password="Heeyoun1220!", dsn="stoxle_high"
        )

    def save_one(self, news: dict, cursor, connection, table: str) -> bool:
        """처리 완료된 기사 1건을 즉시 삽입."""
        insert_sql = f"""
            INSERT INTO {table}
                (link, category, title, summary, sentiment,
                 pub_date, clickbait_prob, article_type, type_prob)
            VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
        """
        try:
            cursor.execute(insert_sql, [
                news['link'], news['category'], news['title'],
                news['summary'], news['sentiment'], news['date'],
                str(news['clickbait_prob']),
                news['article_type'], str(news['type_prob'])
            ])
            connection.commit()
            return True
        except oracledb.IntegrityError:
            return False  # 중복
        except Exception as e:
            print(f"  삽입 에러 [{news['title'][:10]}]: {e}")
            return False